In [3]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"

!wget -nc {PREFIX}/rag_helper.py
!wget -nc {PREFIX}/starter.py

File ‘rag_helper.py’ already there; not retrieving.



File ‘starter.py’ already there; not retrieving.



In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import starter

len(starter.documents)

72

In [6]:
import time
import uuid
import statistics
from dataclasses import dataclass, field

import pandas as pd
from rag_helper import RAGBase

In [7]:
def calculate_cost(model, usage):
    """Same formula as 05-monitoring lesson 04 (metrics.py)."""
    cost = 0
    if "gpt-5.4-mini" in model:
        cost = (usage.input_tokens * 0.15 + usage.output_tokens * 0.60) / 1_000_000
    return cost

In [8]:
# --- Tracing primitives -----------------------------------------------
# Not course-provided code -- 05-monitoring/lessons never builds a tracer,
# it only names OpenTelemetry as the standard behind commercial tools
# (lesson 14). This is a minimal, dependency-free version in the same
# style as the course's own LLMCallRecord: a plain dataclass per span,
# no external tracing library.

@dataclass
class Span:
    name: str
    start_time: float
    end_time: float = None
    attributes: dict = field(default_factory=dict)

    @property
    def duration_ms(self):
        return (self.end_time - self.start_time) * 1000


@dataclass
class Trace:
    trace_id: str
    spans: list = field(default_factory=list)

    def span_table(self):
        rows = []
        for s in self.spans:
            row = {"name": s.name, "duration_ms": round(s.duration_ms, 1)}
            row.update(s.attributes)
            rows.append(row)
        return rows


class SpanRecorder:
    """Context manager that times a block and appends a Span to a Trace.

    Appends the span to `trace.spans` on __enter__ (not __exit__), so
    spans land in the table in the order they *started* -- parent first,
    then children -- rather than the order they finished.

    This also sidesteps the pitfall flagged in
    05-monitoring/notes/03_common_pitfalls.md #1: RAGWithMetrics.last_call
    is shared mutable state on the instance -- two overlapping calls would
    fight over it. Spans are returned as part of an explicit Trace object
    instead, not stashed on self.
    """

    def __init__(self, trace, name):
        self.trace = trace
        self.name = name
        self.span = None

    def __enter__(self):
        self.span = Span(name=self.name, start_time=time.time())
        self.trace.spans.append(self.span)
        return self.span

    def __exit__(self, exc_type, exc, tb):
        self.span.end_time = time.time()
        return False

In [9]:
# --- Traced RAG pipeline ------------------------------------------------
# search() and llm() are used exactly as the real cohort RAGBase already
# defines them -- nothing overridden, nothing duplicated. rag_traced()
# just re-runs the same three steps rag() does, with a span wrapped
# around each one.

class TracedRAG(RAGBase):

    def rag_traced(self, query):
        trace = Trace(trace_id=str(uuid.uuid4()))

        with SpanRecorder(trace, "rag") as rag_span:
            with SpanRecorder(trace, "search") as search_span:
                search_results = self.search(query)
            search_span.attributes["num_results"] = len(search_results)

            prompt = self.build_prompt(query, search_results)

            with SpanRecorder(trace, "llm") as llm_span:
                response = self.llm(prompt)  # real rag_helper.py's llm() returns the raw response

            usage = response.usage
            llm_span.attributes.update({
                "model": self.model,
                "input_tokens": usage.input_tokens,
                "output_tokens": usage.output_tokens,
                "total_tokens": usage.total_tokens,
                "cost": calculate_cost(self.model, usage),
            })

            answer = response.output_text
            rag_span.attributes["answer_length"] = len(answer)

        return answer, trace

In [10]:
assistant = TracedRAG(index=starter.index, llm_client=starter.client)

In [11]:
query = "How does the agentic loop keep calling the model until it stops?"
answer, trace = assistant.rag_traced(query)

print(answer)
print()
pd.DataFrame(trace.span_table())

It keeps calling the model inside a `while True` loop.

Each iteration:
1. Sends the full message history to the model.
2. Checks the response for any `function_call` items.
3. Runs the tool if needed and appends the tool output to `messages`.
4. Sets `has_function_calls = True` if it saw any tool calls.

At the end of the iteration, it breaks only when `has_function_calls == False`, meaning the model returned a final answer with no more tool calls.

So the stopping condition is: **no function calls this turn**.



,name,duration_ms,answer_length,num_results,model,input_tokens,output_tokens,total_tokens,cost
0,rag,7069.3,517.0,NaN,NaN,NaN,NaN,NaN,NaN
1,search,108.9,NaN,5.0,NaN,NaN,NaN,NaN,NaN
2,llm,6960.3,NaN,NaN,gpt-5.4-mini,7111.0,123.0,7234.0,0.00114


In [12]:
# Q1: how many spans does the trace produce?
num_spans = len(trace.spans)
print("Q1 - number of spans:", num_spans)

# Q2: input tokens on the llm span
llm_span = next(s for s in trace.spans if s.name == "llm")
input_tokens = llm_span.attributes["input_tokens"]
print("Q2 - llm span input_tokens:", input_tokens)

# Q3: roughly how long does the llm call take
llm_duration_ms = llm_span.duration_ms
print("Q3 - llm span duration_ms:", round(llm_duration_ms, 1))

# Q4: which span names appear in the spans table
span_names = [s.name for s in trace.spans]
print("Q4 - span names:", span_names)

# Q5: excluding the rag span, which span type takes the most total time
non_rag = [s for s in trace.spans if s.name != "rag"]
by_duration = sorted(non_rag, key=lambda s: s.duration_ms, reverse=True)
print("Q5 - span durations excluding rag:",
      [(s.name, round(s.duration_ms, 1)) for s in by_duration])

Q1 - number of spans: 3
Q2 - llm span input_tokens: 7111
Q3 - llm span duration_ms: 6960.3
Q4 - span names: ['rag', 'search', 'llm']
Q5 - span durations excluding rag: [('llm', 6960.3), ('search', 108.9)]


In [13]:
# Q6: how much do input tokens vary across 4 runs of the same query?
runs = []
for i in range(4):
    _, run_trace = assistant.rag_traced(query)
    run_llm_span = next(s for s in run_trace.spans if s.name == "llm")
    runs.append(run_llm_span.attributes["input_tokens"])

print("Q6 - input_tokens across 4 runs:", runs)

mean_tokens = statistics.mean(runs)
max_dev = max(abs(t - mean_tokens) for t in runs)
pct_variance = (max_dev / mean_tokens) * 100 if mean_tokens else 0
print(f"Q6 - max deviation from mean: {pct_variance:.2f}%")

Q6 - input_tokens across 4 runs: [7111, 7111, 7111, 7111]
Q6 - max deviation from mean: 0.00%


In [14]:
print("=== Answers summary ===")
print("Q1 (span count):        ", num_spans)
print("Q2 (llm input tokens):  ", input_tokens)
print("Q3 (llm duration ms):   ", round(llm_duration_ms, 1))
print("Q4 (span names):        ", span_names)
print("Q5 (search vs llm, ms): ", [(s.name, round(s.duration_ms, 1)) for s in by_duration])
print("Q6 (token runs, %dev):  ", runs, f"{pct_variance:.2f}%")

=== Answers summary ===
Q1 (span count):         3
Q2 (llm input tokens):   7111
Q3 (llm duration ms):    6960.3
Q4 (span names):         ['rag', 'search', 'llm']
Q5 (search vs llm, ms):  [('llm', 6960.3), ('search', 108.9)]
Q6 (token runs, %dev):   [7111, 7111, 7111, 7111] 0.00%
